# Paper 3 environmental sound analysis: YAMNet + frequency

Audio is divided into exact 10-second windows aligned to `key.csv`, with each recording anchored to the timestamp in its WAV filename. The notebook joins UCM `ucm_SND_dBA` from the full `merged_all_11participants.csv`, calculates frequency descriptors from the original WAV sample rate, and uses exact official YAMNet/AudioSet class names to sum class mean scores within each researcher category. The largest category total determines `category` when it reaches the 0.20 cutoff; otherwise, `category` is `Unidentify`. The seven category totals are exported as `*_score`. The eight `*_proportion` fields measure overlapping YAMNet frame detections at 0.20. The final reduced CSVs retain 38 columns: the category, seven category scores, eight category proportions, 15 frequency descriptors and seven identification/level fields. Individual YAMNet class columns are omitted from reduced exports.

**References (what the frequency columns are grounded in)**

Calibrated sound level is supplied by the UCM backpack (`ucm_SND_dBA`). YAMNet provides scores for AudioSet classes (Gemmeke et al., 2017, [doi:10.1109/ICASSP.2017.7952131](https://doi.org/10.1109/ICASSP.2017.7952131)); the category mapping, score aggregation and 0.20 cutoff are study choices. The three frequency ranges (20?200, 200?2000 and 2000?20000 Hz) are reported by Di, Lu & Shi (2018, [doi:10.3390/ijerph15030474](https://doi.org/10.3390/ijerph15030474), Eq. 8) and Yang et al. (2024, [doi:10.3390/app14010175](https://doi.org/10.3390/app14010175), Section 2.2.1). Di et al. define relative band energy; here it is estimated by integrating Welch PSD over each band and dividing by total recorded power over 20?20000 Hz. Welch PSD follows Welch (1967, [doi:10.1109/TAU.1967.1161901](https://doi.org/10.1109/TAU.1967.1161901)). Octave summaries use the base-2 definition: centres `1000 ? 2^n` Hz and edges `centre / sqrt(2)` and `centre ? sqrt(2)`. These are relative spectral-power estimates, not calibrated octave-band SPL or an IEC filter-conformance measurement. Hamida et al. (2024, [doi:10.1016/j.indenv.2024.100066](https://doi.org/10.1016/j.indenv.2024.100066)) provide the level, type and frequency framing, rather than prescribing this implementation.

The primary sound-exposure variable is UCM `ucm_SND_dBA`. YAMNet categories and frequency descriptors are supplementary outputs.

Shared class memberships are preserved from the category dictionary; ties follow dictionary order. The existing 0.20 cutoff is provisional when applied to category sums. A missing category means inference was unavailable; it is not `Unidentify`. Category totals are aggregate model scores, not calibrated probabilities.


In [1]:
"""
Sound analysis for Paper 3.

The pipeline keeps three acoustic aspects separate:
1) Sound level: UCM SND_dBA is the primary calibrated environmental
   sound-level measure. WAV RMS dBFS is calculated only as a validity and
   non-silent-window check; it is not exported as a primary sound-level field.
2) Sound source: YAMNet runs on 16 kHz mono audio. Its class-level outputs
   use the official 521-class YAMNet/AudioSet class map. Dominant-class,
   frame-count, and threshold fields are derived metadata. Researcher
   categories are generated from the mapped YAMNet classes, with one frame
   proportion for each category retained in the final reduced outputs.
3) Sound frequency: Welch power spectral density is calculated from the
   original sample-rate WAV in the same 10-second windows.

Hamida et al. (2024; doi:10.1016/j.indenv.2024.100066) argue for separating sound
level, type, and frequency; they do not define the Hz bins used here.
Three broad bands (20-200, 200-2000, 2000-20000 Hz) use the ranges reported by
Di, Lu & Shi (2018; doi:10.3390/ijerph15030474) and Yang et al. (2024;
doi:10.3390/app14010175). Di et al., Eq. 8, defines relative band energy. Here,
each proportion is integrated Welch PSD in that band divided by recorded
spectral power over 20-20000 Hz; it is not calibrated band sound pressure level.
Octave summaries use base-2 centres (1000 * 2^n Hz) and edges centre/sqrt(2)
to centre*sqrt(2); no IEC filter conformance is claimed. Welch PSD follows
Welch (1967; doi:10.1109/TAU.1967.1161901). YAMNet threshold and windowing are
pipeline choices for this study.

Checks applied during the primary analysis:
- Each participant-phase must have exactly one WAV file.
- Audio phase bounds come from key.csv and are measured from the timestamp in the WAV filename. The UCM dBA grid uses the Brussels-local timestamps produced by 02_ucm_build.py (+2 h), and is checked against that same phase start after flooring to the shared 10-second clock grid.
- The full combined output retains a shorter final window with is_complete_10s_window=False for audit/QC. Its YAMNet and frequency descriptors use only the real available audio, without padding. The matched 8-minute output contains complete 10-second windows only.
- No per-file amplitude normalization is applied.
- Frequency analysis uses the original waveform, not the 16-kHz YAMNet input.
- Spectral outputs are relative energy descriptors, not calibrated dB SPL.
"""

from __future__ import annotations

import argparse
import csv
import io
import math
import os
import re
import tempfile
from dataclasses import dataclass
from datetime import datetime, timedelta
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf
import tensorflow_hub as hub
from scipy.integrate import trapezoid
from scipy.signal import resample_poly, welch


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

SEGMENT_DURATION_S = 10.0
YAMNET_SR = 16_000
YAMNET_THRESHOLD = 0.20
AUDIO_MIN_HZ = 20.0
AUDIO_MAX_HZ = 20_000.0

DEFAULT_PROJECT_ROOT = Path(r"C:\Users\pandya\Documents\Github\docker\Paper3_Final")
DEFAULT_RAW_UCM_ROOT = Path(
    r"C:\Users\pandya\OneDrive - UCL\Field experiment raw data\Complete Participantwise data\Final Data\ucm"
)
DEFAULT_KEY_CSV = DEFAULT_PROJECT_ROOT / "output" / "key.csv"
DEFAULT_MERGED_CSV = DEFAULT_PROJECT_ROOT.parent / "Paper3_Github" / "output" / "merged_all_11participants.csv"
DEFAULT_OUTPUT_DIR = DEFAULT_PROJECT_ROOT / "output" / "Sound_ucm"

RETAINED_PARTICIPANTS = {"P4", "P8", "P9", "P10", "P11", "P12", "P13", "P14", "P15", "P16", "P17"}
PHASES = ["BikeG", "BikeU", "Tram", "WalkG", "WalkU"]
# Run the retained 11-participant cohort across all five route phases.
RUN_PARTICIPANTS = RETAINED_PARTICIPANTS
RUN_PHASES = PHASES
PRIMARY_MATCHED_SECONDS = 480.0
UCM_LOCAL_TIME_OFFSET = timedelta(hours=2)

# Categories retained from the original JOS notebook.
# The pipe delimiter preserves commas that are part of AudioSet labels.
RESEARCHER_CATEGORIES = {
    'Wind': 'Wind|Wind noise (microphone)|Howl|Whoosh, swoosh, swish|Gust of wind|Blowing (wind)|Storm|Hurricane|Tornado'.split('|'),
    'Rural/Natural': 'Outside, rural or natural|Forest|Bird|Bird vocalization, bird call, bird song|Chirp, tweet|Coo|Crow|Insect|Cricket|Bee, wasp, etc.|Frog|Nature sounds|Stream|Water|Rain|Thunderstorm|Thunder|Ocean|Waves, surf|Animal|Livestock, farm animals, working animals|Horse|Cattle, bovinae|Sheep|Goat|Pig|Chicken, rooster|Dog|Cat|Wild animals|Wolf|Bird flight, flapping wings'.split('|'),
    'Vehicle': 'Vehicle|Motor vehicle (road)|Car|Car passing by|Truck|Bus|Motorcycle|Emergency vehicle|Siren|Police car (siren)|Ambulance (siren)|Fire engine, fire truck (siren)|Aircraft|Airplane|Helicopter|Train|Rail transport|Train horn|Subway, metro, underground|Boat, Water vehicle|Ship|Motorboat, speedboat|Engine|Engine starting|Idling|Accelerating, revving, vroom|Car alarm|Vehicle horn, car horn, honking'.split('|'),
    'Urban/Manmade_neutral': 'Outside, urban or manmade|City|Urban|Footsteps|Walking|Running|Skateboard|Bicycle'.split('|'),
    'Urban/Manmade_positive': 'Crowd|Cheering|Applause|Chatter|Conversation|Speech|Human voice|Street music|Busker|Music'.split('|'),
    'Urban/Manmade_negative': 'Traffic noise, roadway noise|Construction site|Jackhammer|Bulldozer|Excavator|Drilling|Hammering|Sawing|Power tool|Chainsaw|Siren|Emergency vehicle|Industrial noise|Factory|Machinery|Air conditioning'.split('|'),
    'Mechanisms': 'Mechanisms|Tools|Hammer|Hammering|Wood|Sawing|Filing (rasp)|Sanding|Power tool|Drill|Jackhammer|Chainsaw|Medium engine (mid frequency)|Light engine (high frequency)|Heavy engine (low frequency)|Engine knocking|Engine starting|Idling|Accelerating, revving, vroom|Door|Sliding door|Slam|Knock|Tap|Squeak|Creak|Ratchet, pawl|Clock|Tick|Ticking|Mechanical fan|Air conditioning|Microwave oven|Blender|Water tap, faucet|Flush|Zipper (clothing)|Keys jangling|Coin (dropping)|Scissors|Electric shaver, electric razor|Lawn mower|Printing|Typewriter|Keyboard (typing)'.split('|'),
}
RESEARCHER_CATEGORY_CODES = {
    'Unidentify': 0, 'Wind': 1, 'Rural/Natural': 2, 'Vehicle': 3,
    'Urban/Manmade_neutral': 4, 'Urban/Manmade_positive': 5,
    'Urban/Manmade_negative': 6, 'Mechanisms': 7,
}

RESEARCHER_CATEGORY_PROPORTION_FIELDS = {
    'Wind': 'wind_proportion',
    'Rural/Natural': 'rural_natural_proportion',
    'Vehicle': 'vehicle_proportion',
    'Urban/Manmade_neutral': 'urban_manmade_neutral_proportion',
    'Urban/Manmade_positive': 'urban_manmade_positive_proportion',
    'Urban/Manmade_negative': 'urban_manmade_negative_proportion',
    'Mechanisms': 'mechanisms_proportion',
    'Unidentify': 'unidentify_proportion',
}

RESEARCHER_CATEGORY_SCORE_FIELDS = {
    category: field.removesuffix('_proportion') + '_score'
    for category, field in RESEARCHER_CATEGORY_PROPORTION_FIELDS.items()
    if category != 'Unidentify'
}

# Final CSVs retain researcher categories; individual YAMNet class fields are omitted.

# 1/1-octave centres f_c = 1000 * 2^n Hz (n = -5..3); band edges f_c/sqrt(2) to f_c*sqrt(2)
# Only complete bands within the analysed frequency range are reported.
OCTAVE_CENTERS_HZ = [1000.0 * 2.0**n for n in range(-5, 4)]

# Three-band partition (Di, Lu & Shi 2018 doi:10.3390/ijerph15030474 Eq. 8;
# Yang et al. 2024 doi:10.3390/app14010175 Sec. 2.2.1). Field: Welch energy share.
DI_YANG_BROAD_BANDS = [
    ("freq_proportion_20_200", 20.0, 200.0),
    ("freq_proportion_200_2000", 200.0, 2000.0),
    ("freq_proportion_2000_20000", 2000.0, 20000.0),
]
FREQ_PROPORTION_COLUMNS = [key for key, _, _ in DI_YANG_BROAD_BANDS]
OCTAVE_PROPORTION_COLUMNS = [f"octave_proportion_{center:g}hz" for center in OCTAVE_CENTERS_HZ]
SPECTRAL_DESCRIPTOR_COLUMNS = [
    "dominant_frequency_hz", "spectral_centroid_hz", "spectral_rolloff85_hz",
]
CATEGORY_PROPORTION_COLUMNS = list(RESEARCHER_CATEGORY_PROPORTION_FIELDS.values())
CATEGORY_SCORE_COLUMNS = list(RESEARCHER_CATEGORY_SCORE_FIELDS.values())

# ---------------------------------------------------------------------------
# YAMNet model and class map
# ---------------------------------------------------------------------------

def load_yamnet():
    model = hub.load("https://tfhub.dev/google/yamnet/1")
    class_map_path = model.class_map_path().numpy().decode("utf-8")
    with tf.io.gfile.GFile(class_map_path, "r") as f:
        rows = list(csv.reader(io.StringIO(f.read())))
    class_names = [row[2] for row in rows if len(row) >= 3 and row[0] != "index"]
    if len(class_names) != 521:
        raise RuntimeError(f"Expected 521 YAMNet classes; got {len(class_names)}")
    for category, labels in RESEARCHER_CATEGORIES.items():
        missing = sorted(set(labels).difference(class_names))
        if missing:
            print(f"Mapping notice: {category} labels absent from YAMNet; excluded from scoring: {missing}")
        if not set(labels).intersection(class_names):
            raise RuntimeError(f"No exact official YAMNet labels found for category {category}")
    return model, class_names


def _official_class_slug(label: str) -> str:
    slug = re.sub(r"[^0-9A-Za-z]+", "_", label).strip("_").lower()
    return slug or "unnamed"


def build_official_class_fields(class_names: list[str]) -> list[str]:
    # Each output field keeps the official class-map index and label.
    fields = [
        f"yamnet_class_{i:03d}_{_official_class_slug(label)}"
        for i, label in enumerate(class_names)
    ]
    if len(fields) != 521 or len(set(fields)) != 521:
        raise RuntimeError("Official YAMNet class output fields are not a unique 521-class schema")
    return fields


def classify_researcher_category(
    class_names: list[str],
    class_means: np.ndarray,
    frame_scores: np.ndarray,
    threshold: float,
) -> tuple[str, int, dict[str, float], dict[str, float]]:
    """Select the researcher category with the largest sum of class mean scores."""
    class_means = np.asarray(class_means, dtype=float)
    frame_scores = np.asarray(frame_scores, dtype=float)
    if (frame_scores.ndim != 2 or frame_scores.shape[0] == 0
            or frame_scores.shape[1] != len(class_names)
            or class_means.shape != (len(class_names),)):
        raise ValueError("Category scores require non-empty frames and matching class dimensions")
    if (not np.isfinite(frame_scores).all() or not np.isfinite(class_means).all()
            or ((frame_scores < 0) | (frame_scores > 1)).any()
            or ((class_means < 0) | (class_means > 1)).any()
            or not 0 <= threshold <= 1):
        raise ValueError("YAMNet class scores and detection threshold must be finite and within [0, 1]")
    if len(set(class_names)) != len(class_names):
        raise ValueError("YAMNet class names must be unique")

    category_scores = {}
    category_proportions = {category: 0.0 for category in RESEARCHER_CATEGORY_PROPORTION_FIELDS}
    any_researcher_match = np.zeros(frame_scores.shape[0], dtype=bool)
    for category, labels in RESEARCHER_CATEGORIES.items():
        # Exact official labels prevent Train/Rain, Crowd/Crow and Fire/Fire engine matches.
        label_set = set(labels)
        indices = [i for i, label in enumerate(class_names) if label in label_set]
        category_scores[category] = float(class_means[indices].sum())
        if indices:
            active = np.any(frame_scores[:, indices] >= threshold, axis=1)
            category_proportions[category] = float(active.mean())
            any_researcher_match |= active

    # Preserve listed overlapping memberships; equal totals use dictionary order.
    best_category = max(category_scores, key=category_scores.get)
    # shortcut: 0.20 is an operational sum-score cutoff, validate with labelled audio before final reporting.
    if category_scores[best_category] < threshold or category_scores[best_category] == 0:
        best_category = 'Unidentify'
    category_proportions['Unidentify'] = float((~any_researcher_match).mean())
    return best_category, RESEARCHER_CATEGORY_CODES[best_category], category_proportions, category_scores


# ---------------------------------------------------------------------------
# Audio and timing
# ---------------------------------------------------------------------------

def load_original_audio_mono(
    path: Path,
    stereo_strategy: str = "channel_0",
    start_frame: int = 0,
    stop_frame: int | None = None,
) -> tuple[np.ndarray, int, int, str]:
    # Read only the requested window; some field WAVs are too large for a
    # full float32 allocation in memory.
    info = sf.info(path)
    sr = int(info.samplerate)
    total_frames = int(info.frames)
    if stop_frame is None:
        stop_frame = total_frames
    start_frame = max(0, min(int(start_frame), total_frames))
    stop_frame = max(start_frame, min(int(stop_frame), total_frames))
    with sf.SoundFile(path) as sound_file:
        sound_file.seek(start_frame)
        audio = sound_file.read(
            stop_frame - start_frame, dtype="float32", always_2d=True
        )
    if sr <= 0:
        raise ValueError(f"Invalid sample rate in {path}: {sr}")
    if not np.all(np.isfinite(audio)):
        raise ValueError(f"Non-finite audio samples in {path}")

    channels = int(audio.shape[1])
    if channels == 1:
        mono = audio[:, 0]
        channel_strategy = "native_mono"
    else:
        reference = audio[:, [0]]
        max_channel_difference = float(np.max(np.abs(audio - reference)))
        if max_channel_difference <= 1e-6:
            mono = audio[:, 0]
            channel_strategy = "duplicated_channels_first_channel"
        elif stereo_strategy == "channel_0":
            # Primary capsule / left channel (standard single-channel acquisition)
            mono = audio[:, 0]
            channel_strategy = "stereo_primary_channel_0"
        elif stereo_strategy == "channel_1":
            # Secondary capsule / right channel
            mono = audio[:, 1]
            channel_strategy = "stereo_secondary_channel_1"
        elif stereo_strategy == "average":
            # Linear mid-channel downmix (0.5 * (L + R))
            mono = 0.5 * (audio[:, 0] + audio[:, 1])
            channel_strategy = "stereo_linear_average"
        else:
            raise RuntimeError(
                f"{path} has {channels} non-identical channels (max abs diff = {max_channel_difference:.6f}). "
                "Specify --stereo-strategy (channel_0, channel_1, or average)."
            )

    # soundfile scales integer PCM to [-1, 1]. Keep the original waveform for
    # RMS and PSD; do not clip or normalize it here. RMS is used only as a
    # validity/non-silent gate and is not exported as a sound-level field.
    peak = float(np.max(np.abs(mono))) if mono.size else 0.0
    if peak > 1.0001:
        raise ValueError(
            f"{path} has |sample|={peak:.6f} > 1.0; dBFS/spectral interpretation "
            "requires verification of the WAV encoding before analysis"
        )
    return np.asarray(mono, dtype=np.float32), int(sr), channels, channel_strategy


def to_yamnet_16k(segment: np.ndarray, sr: int) -> np.ndarray:
    data = np.asarray(segment, dtype=np.float32)
    if sr == YAMNET_SR:
        out = data
    else:
        # Use polyphase resampling so the conversion does not depend on optional
        # codec or resampler packages.
        g = math.gcd(int(sr), int(YAMNET_SR))
        up = YAMNET_SR // g
        down = int(sr) // g
        out = resample_poly(data, up, down).astype(np.float32)
    return np.clip(out, -1.0, 1.0)


def rms_dbfs(segment: np.ndarray) -> float:
    if segment.size == 0:
        return np.nan
    rms = float(np.sqrt(np.mean(np.square(segment, dtype=np.float64))))
    if not np.isfinite(rms) or rms <= 0.0:
        return np.nan
    return float(20.0 * np.log10(rms))


def parse_time_str(value):
    if pd.isna(value):
        return None
    return datetime.strptime(str(value).strip(), "%H:%M:%S").time()


@dataclass(frozen=True)
class Recording:
    participant: str
    phase: str
    wav_path: Path
    phase_start_offset_s: float
    active_duration_s: float
    alignment_source: str


def discover_recordings(
    raw_root: Path,
    key_csv: Path,
    participants: Iterable[str] = RETAINED_PARTICIPANTS,
    phases: Iterable[str] = PHASES,
) -> list[Recording]:
    if not raw_root.is_dir():
        raise FileNotFoundError(raw_root)
    if not key_csv.is_file():
        raise FileNotFoundError(key_csv)

    key_df = pd.read_csv(key_csv)
    required_key = {"Participant_ID"}
    missing = required_key.difference(key_df.columns)
    if missing:
        raise ValueError(f"key.csv missing required columns: {sorted(missing)}")
    key_df = key_df.copy()
    key_df["pid_clean"] = key_df["Participant_ID"].apply(lambda x: f"P{int(x)}")

    selected_pids = set(participants)
    selected_phases = list(phases)
    found: list[Recording] = []
    for pid in sorted(selected_pids, key=lambda x: int(x[1:])):
        pid_path = raw_root / pid
        if not pid_path.is_dir():
            raise FileNotFoundError(f"Missing participant folder: {pid_path}")

        k_rows = key_df[key_df["pid_clean"] == pid]
        if len(k_rows) != 1:
            raise ValueError(f"Expected exactly one key.csv row for {pid}; got {len(k_rows)}")
        k_row = k_rows.iloc[0]

        for phase in selected_phases:
            phase_dir = pid_path / phase
            if not phase_dir.is_dir():
                raise FileNotFoundError(f"Missing phase folder: {phase_dir}")

            wavs = sorted(phase_dir.rglob("*.wav"))
            if len(wavs) != 1:
                raise RuntimeError(
                    f"{pid} {phase}: expected exactly one WAV, found {len(wavs)}: {wavs}"
                )
            wav_path = wavs[0]

            stamp = wav_path.stem
            try:
                wav_dt = datetime.strptime(stamp, "%Y%m%d_%H%M%S")
            except ValueError as exc:
                raise ValueError(f"Cannot parse WAV timestamp: {wav_path.name}") from exc

            k_s = parse_time_str(k_row.get(f"{phase}_start"))
            k_e = parse_time_str(k_row.get(f"{phase}_end"))
            if k_s is None or k_e is None:
                raise ValueError(f"{pid} {phase}: missing {phase}_start/end in key.csv")

            start_dt = datetime.combine(wav_dt.date(), k_s)
            end_dt = datetime.combine(wav_dt.date(), k_e)
            if end_dt <= start_dt:
                end_dt += timedelta(days=1)
            offset_s = (start_dt - wav_dt).total_seconds()
            if offset_s < -1.0:
                raise RuntimeError(
                    f"{pid} {phase}: key phase start precedes WAV start by {abs(offset_s):.1f}s; "
                    "audio alignment is invalid"
                )
            offset_s = max(offset_s, 0.0)
            duration_s = (end_dt - start_dt).total_seconds()
            if duration_s <= 0:
                raise RuntimeError(f"{pid} {phase}: non-positive phase duration {duration_s}")

            found.append(
                Recording(
                    participant=pid,
                    phase=phase,
                    wav_path=wav_path,
                    phase_start_offset_s=float(offset_s),
                    active_duration_s=float(duration_s),
                    alignment_source="key.csv phase bounds relative to WAV filename timestamp",
                )
            )

    expected = len(selected_pids) * len(selected_phases)
    if len(found) != expected:
        raise AssertionError(f"Expected {expected} participant-phase recordings; got {len(found)}")
    return found


# ---------------------------------------------------------------------------
# YAMNet classification
# ---------------------------------------------------------------------------

def classify_yamnet_segment(
    segment_16k: np.ndarray,
    model,
    official_class_fields: list[str],
    class_names: list[str],
    threshold: float,
) -> dict[str, object]:
    scores, _, _ = model(tf.constant(segment_16k, dtype=tf.float32))
    frame_scores = np.asarray(scores.numpy(), dtype=np.float64)
    if frame_scores.ndim != 2 or frame_scores.shape[1] != 521:
        raise RuntimeError(f"Unexpected YAMNet score shape: {frame_scores.shape}")

    official_class_proportions = (frame_scores >= float(threshold)).mean(axis=0)
    official_class_means = frame_scores.mean(axis=0)
    official_class_output = {}
    for i, field in enumerate(official_class_fields):
        official_class_output[f"{field}_mean_score"] = float(official_class_means[i])
        official_class_output[f"{field}_proportion"] = float(official_class_proportions[i])
    researcher_category, researcher_code, researcher_category_proportions, researcher_category_scores = classify_researcher_category(
        class_names, official_class_means, frame_scores, threshold
    )

    # The dominant label is selected directly from the official YAMNet class map.
    official_active = official_class_proportions > 0.0
    if np.any(official_active):
        official_dominant_index = max(
            np.flatnonzero(official_active),
            key=lambda i: (official_class_proportions[i], official_class_means[i], -int(i)),
        )
        official_dominant_index = int(official_dominant_index)
        official_dominant_class = class_names[official_dominant_index]
    else:
        official_dominant_index = -1
        official_dominant_class = None

    result: dict[str, object] = {
        "category": researcher_category,
        "sound_type": researcher_code,
        "yamnet_dominant_class": official_dominant_class,
        "yamnet_dominant_class_index": official_dominant_index,
        "yamnet_frame_count": int(frame_scores.shape[0]),
        "yamnet_threshold": float(threshold),
    }
    result.update({
        RESEARCHER_CATEGORY_PROPORTION_FIELDS[category]: proportion
        for category, proportion in researcher_category_proportions.items()
    })

    result.update({
        RESEARCHER_CATEGORY_SCORE_FIELDS[category]: score
        for category, score in researcher_category_scores.items()
    })

    result.update(official_class_output)

    return result


# ---------------------------------------------------------------------------
# Frequency analysis at the original sample rate
# ---------------------------------------------------------------------------

def _integrate_psd(freqs: np.ndarray, psd: np.ndarray, low: float, high: float) -> float:
    if low < freqs[0] or high > freqs[-1] or high <= low:
        return np.nan
    interior = freqs[(freqs > low) & (freqs < high)]
    grid = np.concatenate(([low], interior, [high]))
    return float(trapezoid(np.interp(grid, freqs, psd), grid))


def spectral_features(segment: np.ndarray, sr: int) -> dict[str, float]:
    result: dict[str, float] = {
        "original_sample_rate_hz": float(sr),
        "frequency_analysis_min_hz": AUDIO_MIN_HZ,
        "frequency_analysis_max_hz": float(min(AUDIO_MAX_HZ, sr / 2.0)),
        "frequency_analysis_full_20khz_coverage": bool((sr / 2.0) >= AUDIO_MAX_HZ),
        **{col: np.nan for col in SPECTRAL_DESCRIPTOR_COLUMNS},
        **{col: np.nan for col in FREQ_PROPORTION_COLUMNS},
        **{col: np.nan for col in OCTAVE_PROPORTION_COLUMNS},
    }

    if segment.size < max(32, int(0.25 * sr)):
        return result
    if not np.any(np.abs(segment) > 0):
        return result

    max_hz = min(AUDIO_MAX_HZ, sr / 2.0)
    if max_hz <= AUDIO_MIN_HZ:
        return result

    nperseg = min(len(segment), max(1024, int(sr)))
    noverlap = nperseg // 2
    freqs, psd = welch(
        segment.astype(np.float64),
        fs=float(sr),
        window="hann",
        nperseg=nperseg,
        noverlap=noverlap,
        detrend="constant",
        scaling="density",
    )
    valid = (freqs >= AUDIO_MIN_HZ) & (freqs <= max_hz) & np.isfinite(psd)
    f = freqs[valid]
    p = psd[valid]
    if f.size < 2 or not np.any(p > 0):
        return result

    total = float(trapezoid(p, f))
    if not np.isfinite(total) or total <= 0:
        return result

    result["dominant_frequency_hz"] = float(f[int(np.argmax(p))])
    result["spectral_centroid_hz"] = float(np.sum(f * p) / np.sum(p))

    cumulative = np.cumsum((p[:-1] + p[1:]) * 0.5 * np.diff(f))
    roll_target = 0.85 * total
    roll_idx = int(np.searchsorted(cumulative, roll_target, side="left"))
    roll_idx = min(roll_idx, len(f) - 1)
    result["spectral_rolloff85_hz"] = float(f[min(roll_idx + 1, len(f) - 1)])

    for key, low, high in DI_YANG_BROAD_BANDS:
        if high <= max_hz and low >= AUDIO_MIN_HZ:
            e = _integrate_psd(f, p, low, high)
            result[key] = float(e / total) if np.isfinite(e) else np.nan

    if max_hz >= AUDIO_MAX_HZ:
        broad_total = sum(result[key] for key, _, _ in DI_YANG_BROAD_BANDS)
        if not np.isclose(broad_total, 1.0, atol=1e-8, rtol=0):
            raise ValueError("Di/Yang broad bands do not partition total energy")

    for center in OCTAVE_CENTERS_HZ:
        key = f"octave_proportion_{center:g}hz"
        low = center / math.sqrt(2.0)
        high = center * math.sqrt(2.0)
        # Do not label a truncated edge band as a complete octave band.
        if low < AUDIO_MIN_HZ or high > max_hz:
            continue
        e = _integrate_psd(f, p, low, high)
        result[key] = float(e / total) if np.isfinite(e) else np.nan

    return result


# ---------------------------------------------------------------------------
# UCM SND_dBA lookup
# ---------------------------------------------------------------------------

def load_ucm_dba_lookup(
    merged_csv: Path,
    participant: str,
    phase: str,
    expected_phase_start: datetime | None = None,
) -> dict[int, float]:
    """Create a segment-index to UCM SND_dBA lookup.

    Segment i starts at i * SEGMENT_DURATION_S. The merged CSV is sampled
    every 10 seconds, so elapsed_s / 10 gives the segment index. The first
    merged timestamp must equal the expected phase start floored to the shared
    10-second grid when an expected start is supplied. Missing or non-finite
    values are omitted; callers receive np.nan when no value is found.
    """
    if not merged_csv.is_file():
        return {}
    try:
        df = pd.read_csv(
            merged_csv,
            usecols=["ParticipantID", "PhaseID", "Datetime", "ucm_SND_dBA"],
            low_memory=False,
        )
    except ValueError as exc:
        return {}
    sub = df[(df["ParticipantID"] == participant) & (df["PhaseID"] == phase)].copy()
    if sub.empty:
        return {}
    sub["Datetime"] = pd.to_datetime(sub["Datetime"], errors="coerce")
    phase_t0 = sub["Datetime"].min()
    if pd.isna(phase_t0):
        return {}
    if expected_phase_start is not None:
        expected_grid_t0 = pd.Timestamp(expected_phase_start).floor("10s")
        if phase_t0 != expected_grid_t0:
            raise RuntimeError(
                f"{participant} {phase}: UCM SND_dBA grid starts at {phase_t0}; "
                f"expected floored key/WAV phase start {expected_grid_t0}"
            )
    sub = sub.dropna(subset=["Datetime", "ucm_SND_dBA"])
    if sub.empty:
        return {}
    sub["elapsed_s"] = (sub["Datetime"] - phase_t0).dt.total_seconds()
    lookup: dict[int, float] = {}
    for _, row in sub.iterrows():
        seg_idx = int(round(row["elapsed_s"] / SEGMENT_DURATION_S))
        val = float(row["ucm_SND_dBA"])
        if np.isfinite(val):
            lookup[seg_idx] = val
    return lookup


# Full per-window CSV from analyze_recording (audit + all 521 YAMNet class columns appended).
OUTPUT_COLUMNS = [
    "participant", "phase", "segment", "date", "time", "start_time_s", "end_time_s",
    "window_duration_s", "is_complete_10s_window", "phase_start_offset_s",
    "original_channel_count", "channel_strategy",
    "ucm_SND_dBA",
    "sound_type", "category",
    *CATEGORY_PROPORTION_COLUMNS,
    *CATEGORY_SCORE_COLUMNS,
    "yamnet_dominant_class",
    "yamnet_dominant_class_index", "yamnet_frame_count", "yamnet_threshold",
    "original_sample_rate_hz", "frequency_analysis_min_hz",
    "frequency_analysis_max_hz", "frequency_analysis_full_20khz_coverage",
    *SPECTRAL_DESCRIPTOR_COLUMNS,
    *FREQ_PROPORTION_COLUMNS,
    *OCTAVE_PROPORTION_COLUMNS,
]

# Paper 3 / H4 exports: subset of OUTPUT_COLUMNS (no YAMNet QC, no per-class scores).
#
# 1. Sound level — ucm_SND_dBA is the calibrated UCM measurement (primary exposure).
# 2. Sound type (categories) — *_score sums mean AudioSet class scores within each
#    researcher category; *_proportion is the fraction of frames with a mapped class
#    score >= 0.20. Categories can overlap. Gemmeke et al. (2017),
#    doi:10.1109/ICASSP.2017.7952131.
# 3. Broad frequency bands — 20-200, 200-2000 and 2000-20000 Hz follow Di et al. (2018),
#    doi:10.3390/ijerph15030474, and Yang et al. (2024), doi:10.3390/app14010175.
#    freq_proportion_* is band power / total 20-20000 Hz power from Welch PSD (Welch, 1967;
#    doi:10.1109/TAU.1967.1161901) on each available audio window.
# 4. Octave bands — octave_proportion_* uses 1/1-octave base-2 centres and sqrt(2) edges
#    (IEC 61260-1:2014; EN 61260-1:2014, doi:10.31030/2231136); values are relative power.
# 5. Spectral descriptors — peak frequency, power-weighted mean frequency, and the
#    frequency below which 85% of analysed power occurs (rolloff).
# 6. Level / type / frequency framing — Hamida et al. (2024), doi:10.1016/j.indenv.2024.100066.
FINAL_ANALYSIS_COLUMNS = [
    "participant", "phase", "segment", "start_time_s", "end_time_s",
    "phase_start_offset_s", "ucm_SND_dBA", "category",
    *CATEGORY_SCORE_COLUMNS,
    *CATEGORY_PROPORTION_COLUMNS,
    *SPECTRAL_DESCRIPTOR_COLUMNS,
    *FREQ_PROPORTION_COLUMNS,
    *OCTAVE_PROPORTION_COLUMNS,
]
assert len(FINAL_ANALYSIS_COLUMNS) == 38
assert set(FINAL_ANALYSIS_COLUMNS).issubset(OUTPUT_COLUMNS)

def official_class_output_columns(official_class_fields: list[str]) -> list[str]:
    return [
        suffix
        for field in official_class_fields
        for suffix in (f"{field}_mean_score", f"{field}_proportion")
    ]


# ---------------------------------------------------------------------------
# Processing pipeline
# ---------------------------------------------------------------------------

def analyze_recording(
    recording: Recording,
    model,
    official_class_fields: list[str],
    class_names: list[str],
    threshold: float,
    stereo_strategy: str = "channel_0",
    merged_csv: Path = DEFAULT_MERGED_CSV,
) -> pd.DataFrame:
    audio_info = sf.info(recording.wav_path)
    sr = int(audio_info.samplerate)
    total_frames = int(audio_info.frames)
    total_duration_s = total_frames / sr

    crop_start = int(round(recording.phase_start_offset_s * sr))
    crop_end = int(round((recording.phase_start_offset_s + recording.active_duration_s) * sr))
    tolerance_samples = int(round(1.0 * sr))
    if crop_start < 0 or crop_start >= total_frames:
        raise RuntimeError(
            f"{recording.participant} {recording.phase}: crop start outside WAV "
            f"(offset={recording.phase_start_offset_s:.2f}s, WAV={total_duration_s:.2f}s)"
        )
    if crop_end > total_frames + tolerance_samples:
        missing_tail_s = (crop_end - total_frames) / sr
        if missing_tail_s > SEGMENT_DURATION_S:
            raise RuntimeError(
                f"{recording.participant} {recording.phase}: requested phase end exceeds WAV by "
                f"{missing_tail_s:.2f}s; refusing to hide a missing full window"
            )
        print(
            f"  Trimming unrecorded trailing tail: {recording.participant} "
            f"{recording.phase} missing {missing_tail_s:.2f}s after WAV end"
        )
    crop_end = min(crop_end, total_frames)
    clipped, sr, channel_count, channel_strategy = load_original_audio_mono(
        recording.wav_path,
        stereo_strategy=stereo_strategy,
        start_frame=crop_start,
        stop_frame=crop_end,
    )

    # Build the segment-index to UCM SND_dBA lookup once per recording.
    wav_dt = datetime.strptime(recording.wav_path.stem, "%Y%m%d_%H%M%S")
    phase_start_dt = wav_dt + timedelta(seconds=recording.phase_start_offset_s)
    calib_lookup = load_ucm_dba_lookup(
        merged_csv, recording.participant, recording.phase,
        expected_phase_start=phase_start_dt + UCM_LOCAL_TIME_OFFSET,
    )
    print(
        f"  UCM SND_dBA lookup: {len(calib_lookup)} entries for "
        f"{recording.participant} {recording.phase}"
    )

    nominal_segment_samples = int(round(SEGMENT_DURATION_S * sr))
    if nominal_segment_samples <= 0:
        raise RuntimeError("Invalid 10-second sample count")

    n_segments = math.ceil(len(clipped) / nominal_segment_samples)
    records: list[dict[str, object]] = []

    for i in range(n_segments):
        s = i * nominal_segment_samples
        e = min(s + nominal_segment_samples, len(clipped))
        seg = clipped[s:e]
        duration = len(seg) / sr
        complete = abs(duration - SEGMENT_DURATION_S) <= (1.0 / sr)

        window_start_dt = phase_start_dt + timedelta(seconds=i * SEGMENT_DURATION_S)
        base: dict[str, object] = {
            "participant": recording.participant,
            "phase": recording.phase,
            "segment": i + 1,
            "date": window_start_dt.strftime("%Y-%m-%d"),
            "time": window_start_dt.strftime("%H:%M:%S"),
            "start_time_s": round(i * SEGMENT_DURATION_S, 6),
            "end_time_s": round(i * SEGMENT_DURATION_S + duration, 6),
            "window_duration_s": round(duration, 6),
            "is_complete_10s_window": bool(complete),
            "phase_start_offset_s": recording.phase_start_offset_s,
            "original_channel_count": int(channel_count),
            "channel_strategy": channel_strategy,
            "ucm_SND_dBA": np.nan,
            "sound_type": np.nan,
            "category": None,
            **{field: np.nan for field in RESEARCHER_CATEGORY_PROPORTION_FIELDS.values()},
            **{field: np.nan for field in RESEARCHER_CATEGORY_SCORE_FIELDS.values()},
        }

        level = rms_dbfs(seg)
        # Read the 10-second UCM SND_dBA value for this segment.
        base["ucm_SND_dBA"] = calib_lookup.get(i, np.nan)

        # Analyse every non-silent window using its real available audio.
        # A shorter final window is never zero-padded; it remains marked
        # is_complete_10s_window=False and is excluded from matched analysis.
        if np.isfinite(level):
            seg16 = to_yamnet_16k(seg, sr)
            if complete:
                # Complete windows should resample to about 160,000 samples.
                target_n = int(round(SEGMENT_DURATION_S * YAMNET_SR))
                if abs(len(seg16) - target_n) > 2:
                    raise RuntimeError(
                        f"Unexpected resampled length {len(seg16)} for complete 10-s window"
                    )
                if len(seg16) < target_n:
                    seg16 = np.pad(seg16, (0, target_n - len(seg16)))
                elif len(seg16) > target_n:
                    seg16 = seg16[:target_n]

            base.update(
                classify_yamnet_segment(
                    seg16, model,
                    official_class_fields, class_names,
                    threshold,
                )
            )
            base.update(spectral_features(seg, sr))
        else:
            # A silent or invalid window cannot support acoustic inference.
            base["yamnet_dominant_class"] = None
            base["yamnet_dominant_class_index"] = np.nan
            for field in official_class_output_columns(official_class_fields):
                base[field] = np.nan
            base.update(spectral_features(np.array([], dtype=np.float32), sr))

        records.append(base)

    df = pd.DataFrame(records)
    return df[OUTPUT_COLUMNS + official_class_output_columns(official_class_fields)]


def rebuild_combined_outputs(
    output_dir: Path,
    participants: Iterable[str] = RETAINED_PARTICIPANTS,
    phases: Iterable[str] = PHASES,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    selected_pids = set(participants)
    selected_phases = list(phases)
    canonical_run = selected_pids == set(RETAINED_PARTICIPANTS) and set(selected_phases) == set(PHASES)
    full_suffix = "" if canonical_run else "_full"
    matched_suffix = ""
    expected = {
        f"{pid}_{phase}_sound_analysis.csv"
        for pid in selected_pids
        for phase in selected_phases
    }
    files = sorted(p for p in output_dir.glob("*_sound_analysis.csv") if p.name in expected)
    if len(files) != len(expected):
        missing = sorted(expected.difference({p.name for p in files}))
        raise AssertionError(f"Expected {len(expected)} per-phase files; got {len(files)}. Missing: {missing}")

    frames = [pd.read_csv(p, low_memory=False) for p in files]
    combined = pd.concat(frames, ignore_index=True)
    full_csv = output_dir / f"all_participants_sound_analysis{full_suffix}.csv"
    combined.to_csv(full_csv, index=False)

    matched = combined[
        combined["phase"].isin(selected_phases)
        & combined["is_complete_10s_window"].eq(True)
        & (pd.to_numeric(combined["end_time_s"], errors="coerce") <= PRIMARY_MATCHED_SECONDS)
    ].copy()

    matched_csv = output_dir / f"all_participants_sound_analysis_matched_8min{matched_suffix}.csv"
    matched.to_csv(matched_csv, index=False)

    if set(combined["participant"].dropna()) != selected_pids:
        raise AssertionError("Combined sound output participant set is incorrect")
    if matched.empty or matched["end_time_s"].max() > PRIMARY_MATCHED_SECONDS:
        raise AssertionError("Matched 8-min sound output is invalid")

    return combined, matched


def run_pipeline(
    raw_root: Path = DEFAULT_RAW_UCM_ROOT,
    key_csv: Path = DEFAULT_KEY_CSV,
    merged_csv: Path = DEFAULT_MERGED_CSV,
    output_dir: Path = DEFAULT_OUTPUT_DIR,
    threshold: float = YAMNET_THRESHOLD,
    participants: Iterable[str] = RETAINED_PARTICIPANTS,
    phases: Iterable[str] = PHASES,
    stereo_strategy: str = "channel_0",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not (0.0 <= threshold <= 1.0):
        raise ValueError("YAMNet threshold must be in [0,1]")

    selected_pids = set(participants)
    selected_phases = list(phases)
    output_dir.mkdir(parents=True, exist_ok=True)
    model, class_names = load_yamnet()
    official_class_fields = build_official_class_fields(class_names)
    recordings = discover_recordings(
        raw_root, key_csv, participants=selected_pids, phases=selected_phases
    )

    # Write to a staging directory and replace canonical files only after all
    # participant-phase analyses and combined exports have completed.
    with tempfile.TemporaryDirectory(prefix=".sound_run_", dir=output_dir) as td:
        stage = Path(td)
        for idx, rec in enumerate(recordings, 1):
            print(
                f"[{idx:02d}/{len(recordings):02d}] {rec.participant} {rec.phase} | "
                f"offset={rec.phase_start_offset_s:.1f}s active={rec.active_duration_s:.1f}s"
            )
            df = analyze_recording(
                rec,
                model,
                official_class_fields,
                class_names,
                threshold,
                stereo_strategy=stereo_strategy,
                merged_csv=merged_csv,
            )
            out = stage / f"{rec.participant}_{rec.phase}_sound_analysis.csv"
            df.to_csv(out, index=False)

        combined, matched = rebuild_combined_outputs(
            stage, participants=selected_pids, phases=selected_phases
        )
        staged_files = sorted(stage.glob("*.csv"))
        expected_count = len(selected_pids) * len(selected_phases) + 2
        if len(staged_files) != expected_count:
            raise AssertionError(
                f"Staging produced {len(staged_files)} CSVs; expected {expected_count}"
            )
        for staged in staged_files:
            os.replace(staged, output_dir / staged.name)

    print(f"Full sound output: {len(combined)} rows")
    print(f"Matched 8-min output: {len(matched)} rows")
    return combined, matched


def export_final_analysis_csvs(
    combined_df: pd.DataFrame,
    matched_8min_df: pd.DataFrame,
    output_dir: Path = DEFAULT_OUTPUT_DIR,
    participants: Iterable[str] = RUN_PARTICIPANTS,
    phases: Iterable[str] = RUN_PHASES,
) -> pd.DataFrame:
    """Write 38-column Paper 3 CSVs (overwrites wide per-phase files from run_pipeline)."""
    if not set(FINAL_ANALYSIS_COLUMNS).issubset(combined_df.columns):
        missing = set(FINAL_ANALYSIS_COLUMNS) - set(combined_df.columns)
        raise AssertionError(f"FINAL_ANALYSIS_COLUMNS missing from pipeline output: {sorted(missing)}")
    reduced = matched_8min_df[FINAL_ANALYSIS_COLUMNS].copy()
    for (participant, phase), phase_frame in combined_df.groupby(["participant", "phase"], sort=True):
        path = output_dir / f"{participant}_{phase}_sound_analysis.csv"
        phase_frame[FINAL_ANALYSIS_COLUMNS].to_csv(path, index=False)
        print(
            "Final reduced output:", path, phase_frame.shape[0],
            "rows x", len(FINAL_ANALYSIS_COLUMNS), "columns",
        )
    full_name = (
        "all_participants_sound_analysis.csv"
        if set(participants) == RETAINED_PARTICIPANTS and set(phases) == set(PHASES)
        else "all_participants_sound_analysis_full.csv"
    )
    for name, frame in {
        full_name: combined_df,
        "all_participants_sound_analysis_matched_8min.csv": matched_8min_df,
    }.items():
        path = output_dir / name
        frame[FINAL_ANALYSIS_COLUMNS].to_csv(path, index=False)
        print("Final reduced output:", path, frame.shape[0], "rows x", len(FINAL_ANALYSIS_COLUMNS), "columns")
    assert not any(c.startswith("yamnet_") for c in reduced.columns)
    assert set(reduced["category"].dropna()).issubset(RESEARCHER_CATEGORY_CODES)
    return reduced


def main() -> None:
    parser = argparse.ArgumentParser(description="Paper 3 YAMNet + original-rate sound-frequency pipeline")
    parser.add_argument("--raw-root", type=Path, default=DEFAULT_RAW_UCM_ROOT)
    parser.add_argument("--key-csv", type=Path, default=DEFAULT_KEY_CSV)
    parser.add_argument("--merged-csv", type=Path, default=DEFAULT_MERGED_CSV,
                        help="merged_all_11participants.csv containing 10-s ucm_SND_dBA")
    parser.add_argument("--output-dir", type=Path, default=DEFAULT_OUTPUT_DIR)
    parser.add_argument("--yamnet-threshold", type=float, default=YAMNET_THRESHOLD)
    parser.add_argument("--participants", nargs="+",
                        default=sorted(RETAINED_PARTICIPANTS, key=lambda x: int(x[1:])))
    parser.add_argument("--phases", nargs="+", default=PHASES,
                        help="Phases to process (default: all 5). E.g. --phases BikeG")
    parser.add_argument(
        "--stereo-strategy",
        choices=["channel_0", "channel_1", "average"],
        default="channel_0",
        help="Strategy for multi-channel audio: channel_0 (primary capsule, default), channel_1, or average (mid mix)",
    )
    args = parser.parse_args()
    run_pipeline(
        raw_root=args.raw_root,
        key_csv=args.key_csv,
        merged_csv=args.merged_csv,
        output_dir=args.output_dir,
        threshold=args.yamnet_threshold,
        participants=args.participants,
        phases=args.phases,
        stereo_strategy=args.stereo_strategy,
    )


In [2]:
# Full cohort: 11 participants x five route phases = 55 recordings.
combined_df, matched_8min_df = run_pipeline(
    participants=RUN_PARTICIPANTS, phases=RUN_PHASES,
    merged_csv=DEFAULT_MERGED_CSV, output_dir=DEFAULT_OUTPUT_DIR,
)

Mapping notice: Wind labels absent from YAMNet; excluded from scoring: ['Blowing (wind)', 'Gust of wind', 'Hurricane', 'Storm', 'Tornado']
Mapping notice: Rural/Natural labels absent from YAMNet; excluded from scoring: ['Forest', 'Nature sounds', 'Wolf']
Mapping notice: Vehicle labels absent from YAMNet; excluded from scoring: ['Airplane']
Mapping notice: Urban/Manmade_neutral labels absent from YAMNet; excluded from scoring: ['City', 'Footsteps', 'Running', 'Urban', 'Walking']
Mapping notice: Urban/Manmade_positive labels absent from YAMNet; excluded from scoring: ['Busker', 'Human voice', 'Street music']
Mapping notice: Urban/Manmade_negative labels absent from YAMNet; excluded from scoring: ['Bulldozer', 'Construction site', 'Drilling', 'Excavator', 'Factory', 'Hammering', 'Industrial noise', 'Machinery']
Mapping notice: Mechanisms labels absent from YAMNet; excluded from scoring: ['Flush', 'Hammering', 'Keyboard (typing)', 'Printing', 'Ticking']
[01/55] P4 BikeG | offset=56.0s acti

In [3]:
# Check the completed output.
qc_cols = [
    'participant', 'phase', 'segment', 'date', 'time', 'start_time_s', 'end_time_s',
    'window_duration_s', 'is_complete_10s_window', 'original_channel_count', 'channel_strategy',
    'ucm_SND_dBA', 'category', *RESEARCHER_CATEGORY_SCORE_FIELDS.values(),
    'dominant_frequency_hz', 'spectral_centroid_hz', 'spectral_rolloff85_hz',
    'freq_proportion_20_200', 'freq_proportion_200_2000', 'freq_proportion_2000_20000'
]
official_output_columns = [c for c in combined_df.columns if c.startswith("yamnet_class_")]
assert len(official_output_columns) == 1042
assert list(combined_df.columns) == OUTPUT_COLUMNS + official_output_columns
assert {'category', 'sound_type'}.issubset(combined_df.columns)
assert set(combined_df['participant'].dropna()) == set(RUN_PARTICIPANTS)
assert set(combined_df['phase'].dropna()) == set(RUN_PHASES)
print('Researcher categories:', combined_df['category'].value_counts(dropna=False).to_dict())
print('Sound-type codes:', combined_df['sound_type'].value_counts(dropna=False).to_dict())
display(matched_8min_df[qc_cols].head(20))
print('Full rows:', len(combined_df))
print('Matched rows:', len(matched_8min_df))
print('Official YAMNet class columns:', len(official_output_columns))
print('Output columns:', len(combined_df.columns))

reduced_analysis_df = export_final_analysis_csvs(combined_df, matched_8min_df)
assert len(reduced_analysis_df.columns) == len(FINAL_ANALYSIS_COLUMNS)


Researcher categories: {'Urban/Manmade_positive': 3152, 'Vehicle': 792, 'Rural/Natural': 786, 'Wind': 512, 'Mechanisms': 296, 'Unidentify': 15, 'Urban/Manmade_neutral': 14}
Sound-type codes: {5: 3152, 3: 792, 2: 786, 1: 512, 7: 296, 0: 15, 4: 14}


,participant,phase,segment,date,time,start_time_s,end_time_s,window_duration_s,is_complete_10s_window,original_channel_count,...,urban_manmade_negative_score,mechanisms_score,dominant_frequency_hz,spectral_centroid_hz,spectral_rolloff85_hz,freq_proportion_20_125,freq_proportion_125_1000,freq_proportion_1000_4000,freq_proportion_4000_8000,freq_proportion_8000_20000
0,P10,BikeG,1,2025-08-12,13:16:08,0.0,10.0,10.0,True,2,...,0.013372,0.135268,22.0,140.331282,143.0,0.835022,0.137739,0.025948,0.000631,0.000660
1,P10,BikeG,2,2025-08-12,13:16:18,10.0,20.0,10.0,True,2,...,0.000637,0.013580,25.0,226.077897,334.0,0.745718,0.212821,0.036898,0.002336,0.002227
2,P10,BikeG,3,2025-08-12,13:16:28,20.0,30.0,10.0,True,2,...,0.020109,0.036916,21.0,125.307684,119.0,0.857047,0.120014,0.021728,0.000694,0.000518
3,P10,BikeG,4,2025-08-12,13:16:38,30.0,40.0,10.0,True,2,...,0.010447,0.011423,25.0,126.388591,147.0,0.808401,0.173035,0.017893,0.000326,0.000345
4,P10,BikeG,5,2025-08-12,13:16:48,40.0,50.0,10.0,True,2,...,0.027935,0.129160,28.0,136.762058,131.0,0.841565,0.130763,0.026308,0.000677,0.000687
5,P10,BikeG,6,2025-08-12,13:16:58,50.0,60.0,10.0,True,2,...,0.080846,0.038813,20.0,217.714066,340.0,0.750470,0.188379,0.059257,0.001175,0.000718
6,P10,BikeG,7,2025-08-12,13:17:08,60.0,70.0,10.0,True,2,...,0.004683,0.030310,27.0,145.058290,148.0,0.825135,0.147452,0.026329,0.000642,0.000442
7,P10,BikeG,8,2025-08-12,13:17:18,70.0,80.0,10.0,True,2,...,0.082603,0.027453,42.0,204.277768,301.0,0.749518,0.191926,0.056965,0.001010,0.000581
8,P10,BikeG,9,2025-08-12,13:17:28,80.0,90.0,10.0,True,2,...,0.008842,0.152630,23.0,146.544733,135.0,0.845030,0.129096,0.021966,0.002223,0.001685
9,P10,BikeG,10,2025-08-12,13:17:38,90.0,100.0,10.0,True,2,...,0.007706,0.385173,20.0,186.786454,189.0,0.789271,0.174292,0.031626,0.003158,0.001654


Full rows: 5567
Matched rows: 2634
Official YAMNet class columns: 1042
Output columns: 1097
Final reduced output: C:\Users\pandya\Documents\Github\docker\Paper3_Final\output\Sound_ucm\P10_BikeG_sound_analysis.csv 49 rows x 40 columns
Final reduced output: C:\Users\pandya\Documents\Github\docker\Paper3_Final\output\Sound_ucm\P10_BikeU_sound_analysis.csv 149 rows x 40 columns
Final reduced output: C:\Users\pandya\Documents\Github\docker\Paper3_Final\output\Sound_ucm\P10_Tram_sound_analysis.csv 60 rows x 40 columns
Final reduced output: C:\Users\pandya\Documents\Github\docker\Paper3_Final\output\Sound_ucm\P10_WalkG_sound_analysis.csv 79 rows x 40 columns
Final reduced output: C:\Users\pandya\Documents\Github\docker\Paper3_Final\output\Sound_ucm\P10_WalkU_sound_analysis.csv 170 rows x 40 columns
Final reduced output: C:\Users\pandya\Documents\Github\docker\Paper3_Final\output\Sound_ucm\P11_BikeG_sound_analysis.csv 61 rows x 40 columns
Final reduced output: C:\Users\pandya\Documents\Github\